1 - Le script permettant de télécharger les données de tous les pays


In [ ]:
import os
import requests
import zipfile
import io

def telecharger_donnees_pays():
    # 1. Créer un dossier qui va contenir tous les fichiers CSV extraits
    dossier_destination = "donnees_banque_mondiale"
    os.makedirs(dossier_destination, exist_ok=True)
    print(f"Dossier '{dossier_destination}' prêt.")

    # 2. Récupérer la liste de tous les pays via l'API de la Banque Mondiale
    print("Récupération de la liste des pays...")
    url_countries = "https://api.worldbank.org/v2/country?format=json&per_page=300"
    reponse_pays = requests.get(url_countries)
    donnees_json = reponse_pays.json()

    # L'API renvoie les données dans le deuxième élément de la liste JSON
    liste_complete = donnees_json[1]

    # Filtrer pour ne garder que les "vrais" pays (exclure les régions géographiques mondiales)
    # Un vrai pays a une région définie (différente de 'NA' - Not Applicable)
    codes_pays = [pays['id'] for pays in liste_complete if pays['region']['id'] != 'NA']

    print(f"{len(codes_pays)} pays identifiés. Début des téléchargements (cela peut prendre quelques minutes)...")

    # 3. Boucler sur chaque pays pour télécharger son fichier ZIP
    for code in codes_pays:
        # URL de l'archive CSV pour un pays spécifique (en français)
        url_zip = f"https://api.worldbank.org/v2/fr/country/{code}?downloadformat=csv"

        try:
            r = requests.get(url_zip)
            if r.status_code == 200:
                # Lire le fichier ZIP directement en mémoire (sans le sauvegarder en .zip sur le disque)
                z = zipfile.ZipFile(io.BytesIO(r.content))
                # Extraire les 3 fichiers (API, Metadata_Indicator, Metadata_Country) dans le dossier
                z.extractall(dossier_destination)
                print(f"Succès : {code}")
            else:
                print(f"Erreur de téléchargement pour le pays {code}")

        except Exception as e:
            print(f"Échec critique pour {code} : {e}")

    print("\n✅ Terminé ! Tous les fichiers CSV ont été extraits dans le dossier.")

# Lancer la fonction
telecharger_donnees_pays()

Dossier 'donnees_banque_mondiale' prêt.
Récupération de la liste des pays...
217 pays identifiés. Début des téléchargements (cela peut prendre quelques minutes)...
Succès : ABW
Succès : AFG
Succès : AGO
Succès : ALB
Succès : AND
Succès : ARE
Succès : ARG
Succès : ARM
Succès : ASM
Succès : ATG
Succès : AUS
Succès : AUT
Succès : AZE
Succès : BDI
Succès : BEL
Succès : BEN
Succès : BFA
Succès : BGD
Succès : BGR
Succès : BHR
Succès : BHS
Succès : BIH
Succès : BLR
Succès : BLZ
Succès : BMU
Succès : BOL
Succès : BRA
Succès : BRB
Succès : BRN
Succès : BTN
Succès : BWA
Succès : CAF
Succès : CAN
Succès : CHE
Succès : CHI
Succès : CHL
Succès : CHN
Succès : CIV
Succès : CMR
Succès : COD
Succès : COG
Succès : COL
Succès : COM
Succès : CPV
Succès : CRI
Succès : CUB
Succès : CUW
Succès : CYM
Succès : CYP
Succès : CZE
Succès : DEU
Succès : DJI
Succès : DMA
Succès : DNK
Succès : DOM
Succès : DZA
Succès : ECU
Succès : EGY
Succès : ERI
Succès : ESP
Succès : EST
Succès : ETH
Succès : FIN
Succès : FJI
Succ

2 -Le script permettant de zipper ces différents dossiers et télécharger

---



In [ ]:
# 1. Compresser le dossier contenant tous les CSV en un fichier ZIP
# (Le point d'exclamation permet d'exécuter une commande Linux dans Colab)
!zip -r donnees_banque_mondiale.zip donnees_banque_mondiale/

# 2. Déclencher le téléchargement du fichier ZIP vers ton ordinateur
from google.colab import files
files.download('donnees_banque_mondiale.zip')

  adding: donnees_banque_mondiale/ (stored 0%)
  adding: donnees_banque_mondiale/Metadata_Country_API_NAM_DS2_fr_csv_v2_30631.csv (deflated 15%)
  adding: donnees_banque_mondiale/Metadata_Country_API_LTU_DS2_fr_csv_v2_18612.csv (deflated 17%)
  adding: donnees_banque_mondiale/Metadata_Indicator_API_PAN_DS2_fr_csv_v2_18481.csv (deflated 79%)
  adding: donnees_banque_mondiale/Metadata_Indicator_API_BWA_DS2_fr_csv_v2_56408.csv (deflated 79%)
  adding: donnees_banque_mondiale/Metadata_Country_API_BFA_DS2_fr_csv_v2_30928.csv (deflated 13%)
  adding: donnees_banque_mondiale/Metadata_Indicator_API_MCO_DS2_fr_csv_v2_37444.csv (deflated 79%)
  adding: donnees_banque_mondiale/Metadata_Country_API_AGO_DS2_fr_csv_v2_88769.csv (deflated 15%)
  adding: donnees_banque_mondiale/Metadata_Indicator_API_MMR_DS2_fr_csv_v2_27158.csv (deflated 78%)
  adding: donnees_banque_mondiale/Metadata_Indicator_API_DOM_DS2_fr_csv_v2_18377.csv (deflated 79%)
  adding: donnees_banque_mondiale/Metadata_Indicator_API_COL_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Le même script pour régrouper les données de chaque pays dans un dossier zippé

In [ ]:
import os
import requests
from google.colab import files

def telecharger_zips_pays():
    # 1. Créer le dossier maître qui va contenir tous les ZIPs des pays
    dossier_master = "donnees_banque_mondiale_zips"
    os.makedirs(dossier_master, exist_ok=True)
    print(f"📁 Dossier maître '{dossier_master}' prêt.")

    # 2. Récupérer la liste de tous les pays
    print("🌍 Récupération de la liste des pays...")
    url_countries = "https://api.worldbank.org/v2/country?format=json&per_page=300"
    reponse_pays = requests.get(url_countries)
    donnees_json = reponse_pays.json()

    # Filtrer pour ne garder que les 217 "vrais" pays
    codes_pays = [pays['id'] for pays in donnees_json[1] if pays['region']['id'] != 'NA']

    print(f"⏳ {len(codes_pays)} pays identifiés. Début des téléchargements (cela prendra environ 1 à 2 minutes)...")

    # 3. Boucler sur chaque pays pour télécharger et sauvegarder son ZIP
    for code in codes_pays:
        url_zip = f"https://api.worldbank.org/v2/fr/country/{code}?downloadformat=csv"

        try:
            r = requests.get(url_zip)
            if r.status_code == 200:
                # Créer le chemin du fichier (ex: donnees_banque_mondiale_zips/CMR.zip)
                chemin_zip_pays = os.path.join(dossier_master, f"{code}.zip")

                # Sauvegarder directement le contenu téléchargé en tant que fichier ZIP
                with open(chemin_zip_pays, 'wb') as f:
                    f.write(r.content)
            else:
                print(f"❌ Erreur de téléchargement pour le pays {code}")

        except Exception as e:
            print(f"❌ Échec critique pour {code} : {e}")

    print("\n✅ Terminé ! Tous les fichiers ZIP par pays sont dans le dossier maître.")

# Lancer la fonction de téléchargement
telecharger_zips_pays()

# 4. Compresser le dossier maître avec une commande système
print("\n📦 Compression du dossier maître en cours...")
!zip -q -r tous_les_pays_archive.zip donnees_banque_mondiale_zips/
print("✅ Archive globale 'tous_les_pays_archive.zip' créée.")

# 5. Déclencher le téléchargement automatique vers ton ordinateur
print("\n⬇️ Lancement du téléchargement vers votre machine...")
files.download('tous_les_pays_archive.zip')

📁 Dossier maître 'donnees_banque_mondiale_zips' prêt.
🌍 Récupération de la liste des pays...
⏳ 217 pays identifiés. Début des téléchargements (cela prendra environ 1 à 2 minutes)...

✅ Terminé ! Tous les fichiers ZIP par pays sont dans le dossier maître.

📦 Compression du dossier maître en cours...
✅ Archive globale 'tous_les_pays_archive.zip' créée.

⬇️ Lancement du téléchargement vers votre machine...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>